# Caso E · 01 EDA ERA5 Xàtiva (mock)

> _Tutorial · Caso de uso: **E — Meteo & solar** · Capa Medallion: **bronce** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Conocer la estructura de ERA5 (mock 30 días Xàtiva) y derivar variables útiles para Caso B (forecast eléctrico) y Caso H (chatbot meteorológico).


## 2. Qué se aprende

- Variables ERA5 horarias: T_air, GHI, viento, lluvia, presión.
- Cómo descargar ERA5 real (CDS API) — fuera del repo.
- Conversiones unitarias: K→°C, J/m²→W/m², m→mm.
- Estacionalidad y diurnal cycle.


## 3. Contexto del caso de uso

ERA5 es el reanálisis climático global del ECMWF. Para Xàtiva tenemos un mock con 30 días horarios; el dataset real cubre desde 1940.


## 4. Relación con CENTINELA+

El dominio `weather_station/xativa/era5_gridpoint` complementa los edificios con variables externas correlacionadas.


## 5. Relación con Medallion

Bronce: NetCDF / mock CSV. Plata: `captia_point` con domain `weather_station`.


## 6. Datos de entrada

`era5_xativa_mock.csv`.


## 7. Schema CAPTIA esperado

Tags: `captia_env=dev`, `domain_id=weather_station`, `site_id=xativa`, `asset_id=era5_gridpoint`, `variable ∈ {temperature_outdoor, solar_irradiance, ...}`.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos mock.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "era5_xativa_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df.head()


## 10. Exploración paso a paso

Estadísticas básicas y diurnal cycle.


In [ ]:
print(df.describe().round(2))
df["hour"] = df["timestamp"].dt.hour
df.groupby("hour")[["t_air_c", "ghi_w_m2"]].mean().round(2)


## 11. Transformación bronce → plata

Notebook siguiente.


## 12. Construcción de capa oro

Notebook 03+04.


## 13. Visualizaciones explicativas

T y GHI horarios.


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 3))
ax1.plot(df["timestamp"], df["t_air_c"], color="#FF5722", label="T")
ax1.set_ylabel("°C")
ax2 = ax1.twinx()
ax2.plot(df["timestamp"], df["ghi_w_m2"], color="#FFC107", label="GHI")
ax2.set_ylabel("W/m²")
plt.title("Diurnal — Xàtiva mock 30 días")
plt.tight_layout()


## 14. Validaciones

GHI nunca > 1100 W/m²; T entre -5 y 45.


In [ ]:
assert df["ghi_w_m2"].between(0, 1100).all()
assert df["t_air_c"].between(-5, 50).all()
print("Rangos OK")


## 15. Errores comunes

1. Confundir GHI con DNI (Direct Normal Irradiance).
2. Asumir que `pressure_hpa` es a nivel del mar (es local).
3. Promediar viento sin orientación (vector vs escalar).


## 16. Ejercicios propuestos

1. Calcula la insolación diaria (kWh/m²/día).
2. Compara `t_air_c` con la curva esperada para Csa.
3. Dibuja la rosa de los vientos.


## 17. Cómo se reutiliza con datos reales

Para descargar ERA5 real: registrarse en CDS, instalar `cdsapi`, ejecutar el script `scripts/era5_download.py` (no incluido).


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `05_case_E_weather_solar/02_bronze_to_silver_weather.ipynb`.
- Documento web del caso: `docs/use-cases/case-e-weather-solar.md`.
